# ESCI data loading

This notebook shows how to use `src.data.load_data` to load and prepare the Amazon ESCI Shopping Queries dataset.

## Setup

Ensure parquet files are in `data/` (or set `data_dir`):
- `shopping_queries_dataset_examples.parquet`
- `shopping_queries_dataset_products.parquet`

In [1]:
import sys
from pathlib import Path

# Project root
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## Load the two datasets separately

Load **examples** (query–product pairs with labels) and **products** (product metadata) so you can inspect them before merging.

In [ ]:
import pandas as pd

DATA_DIR = ROOT / "data"
# If parquets live in a subdir, set: DATA_DIR = ROOT / "data" / "esci-data" / "shopping_queries_dataset"

# 1. Examples: one row per (query, product) with ESCI label and split
df_examples = pd.read_parquet(DATA_DIR / "shopping_queries_dataset_examples.parquet")
print("Examples shape:", df_examples.shape)
print("Columns:", df_examples.columns.tolist())
df_examples.sample(5)

In [ ]:
# 2. Products: one row per product with title, brand, etc.
df_products = pd.read_parquet(DATA_DIR / "shopping_queries_dataset_products.parquet")
print("Products shape:", df_products.shape)
print("Columns:", df_products.columns.tolist())
df_products.sample(5)

In [2]:
from src.data.load_data import load_esci

# Load with defaults: small_version=True, locale="us"
# data_dir defaults to data/esci-data/shopping_queries_dataset; if parquets are in data/, pass data_dir="data"
data_dir = ROOT / "data" if (ROOT / "data" / "shopping_queries_dataset_examples.parquet").exists() else None
df = load_esci(data_dir=data_dir)
df.shape

(601354, 16)

## Columns

The returned DataFrame includes query, product fields, **relevance** (1–4: E=4, S=2, C=3, I=1), and **product_text** (expanded with `[PN]`, `[PBN]`, etc.).

In [3]:
df.columns.tolist()

['example_id',
 'query',
 'query_id',
 'product_id',
 'product_locale',
 'esci_label',
 'small_version',
 'large_version',
 'split',
 'product_title',
 'product_description',
 'product_bullet_point',
 'product_brand',
 'product_color',
 'relevance',
 'product_text']

In [5]:
df[["query_id", "query", "relevance", "product_text"]].sample(10)

,query,relevance,product_text
16,!awnmower tires without rims,1,"[PN] RamPro 10"" All Purpose Utility Air Tires/..."
17,!awnmower tires without rims,4,[PN] MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower ...
18,!awnmower tires without rims,1,[PN] NEIKO 20601A 14.5 inch Steel Tire Spoon L...
19,!awnmower tires without rims,2,[PN] 2PK 13x5.00-6 13x5.00x6 13x5x6 13x5-6 2PL...
20,!awnmower tires without rims,4,[PN] (Set of 2) 15x6.00-6 Husqvarna/Poulan Tir...
21,!awnmower tires without rims,4,[PN] MaxAuto 2 Pcs 16x6.50-8 Lawn Mower Tire f...
22,!awnmower tires without rims,3,[PN] Dr.Roc Tire Spoon Lever Dirt Bike Lawn Mo...
23,!awnmower tires without rims,4,"[PN] MARASTAR 21446-2PK 15x6.00-6"" Front Tire ..."
24,!awnmower tires without rims,2,"[PN] 15x6.00-6"" Front Tire Assembly Replacemen..."
25,!awnmower tires without rims,1,[PN] Honda HRR Wheel Kit (2 Front 44710-VL0-L0...


## Save train/test splits

Pass `save_splits_dir` to write `esci_train.parquet` and `esci_test.parquet` to that directory.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

out_dir = ROOT / "data"
df = load_esci(data_dir=data_dir, save_splits_dir=out_dir)
print(f"Loaded {len(df)} rows; train/test written to {out_dir}")

## Options

- **small_version**: `True` (default) uses Task 1 reduced set; `False` uses full set.
- **locale**: `"us"`, `"es"`, `"jp"`.
- **relevance_map**: Custom ESCI label → score dict (default: E=4, S=2, C=3, I=1).

In [ ]:
df_us = load_esci(data_dir=data_dir, locale="us")
df_es = load_esci(data_dir=data_dir, locale="es")
print(f"US: {len(df_us)} rows, ES: {len(df_es)} rows")